# AI Architecture Guardian v5: GPU run

**Before running:** Settings, then Accelerator **GPU T4 x2**, then **Internet ON**. Add the uploaded dataset with *Add Input*. Then **Run All**.

What this does:
1. CodeBERT + CodeReviewer embeddings of every PR diff (about 15 to 25 min).
2. The heterogeneous GNN, 5 seeds, with the repository holdout (about 20 to 40 min).
3. Zips everything to `/kaggle/working/v5_results.zip`. Download it from the *Output* panel.


In [ ]:
!pip install -q torch_geometric
import torch, torch_geometric
print('torch', torch.__version__, '| PyG', torch_geometric.__version__, '| GPU:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Turn on the GPU accelerator in Settings'

In [ ]:
import gzip, pathlib, shutil, zipfile
INPUT = pathlib.Path('/kaggle/input')
def find(*names):
    for name in names:
        hits = sorted(INPUT.rglob(name))
        if hits:
            return hits[0]
    return None
WORK = pathlib.Path('/kaggle/working/ai_guardian'); OUT = pathlib.Path('/kaggle/working/out')
OUT.mkdir(parents=True, exist_ok=True)
if WORK.exists(): shutil.rmtree(WORK)
# Kaggle sometimes unpacks uploaded archives itself, so accept both forms.
code_zip = find('code.zip')
if code_zip is not None and code_zip.is_file():
    zipfile.ZipFile(code_zip).extractall(WORK)
else:
    src_dir = next(p for p in INPUT.rglob('src') if (p / 'ml' / 'coupling_data.py').is_file())
    shutil.copytree(src_dir.parent, WORK)
packed = find('graphs_v5.pt.gz', 'graphs_v5.pt')
assert packed is not None, 'graphs_v5.pt.gz not found: did you add the dataset with Add Input?'
if packed.suffix == '.gz':
    with gzip.open(packed, 'rb') as src, open('/kaggle/working/graphs_v5.pt', 'wb') as dst:
        shutil.copyfileobj(src, dst)
else:
    shutil.copy(packed, '/kaggle/working/graphs_v5.pt')
DIFFS = find('diffs_v5.jsonl'); SPLIT = find('coupling_split_v5.json')
assert DIFFS and SPLIT, 'diffs_v5.jsonl or coupling_split_v5.json missing from the dataset'
shutil.copy(SPLIT, OUT / 'coupling_split_v5.json')
print(sorted(p.name for p in WORK.iterdir()), DIFFS, SPLIT)

In [ ]:
%cd /kaggle/working/ai_guardian
!python -W ignore scripts/train_coupling_gnn.py --graphs /kaggle/working/graphs_v5.pt --split {SPLIT} --output-dir {OUT}/gnn_v5 --seeds 1 2 3 4 5

In [ ]:
# Second, larger configuration. The one used in the report is chosen by VALIDATION PR-AUC only (never test).
!python -W ignore scripts/train_coupling_gnn.py --graphs /kaggle/working/graphs_v5.pt --split {SPLIT} --output-dir {OUT}/gnn_v5_h128 --seeds 1 2 3 4 5 --hidden-channels 128 --layers 2
import json, statistics
for name in ('gnn_v5', 'gnn_v5_h128'):
    runs = json.load(open(OUT / name / 'metrics.json'))['runs']
    print(name, 'validation PR-AUC mean', round(statistics.mean(r['validation']['pr_auc_average_precision'] for r in runs), 4))

In [ ]:
!python -W ignore scripts/embed_diffs_kaggle.py --diffs {DIFFS} --output-dir {OUT}/embeddings_v5

In [ ]:
import json
print(json.dumps({k: v for k, v in json.load(open(OUT / 'gnn_v5/metrics.json')).items() if k.startswith('test_')}, indent=1))
shutil.make_archive('/kaggle/working/v5_results', 'zip', OUT)
print('Download /kaggle/working/v5_results.zip from the Output panel:',
      round(pathlib.Path('/kaggle/working/v5_results.zip').stat().st_size / 1e6, 1), 'MB')